## Synapse Modeling with PyNS (sPyNS) Tutorial:
Authored by: Abdallah Alashqar (abdallah.j.alashqar@fau.de)

Authored on: 05.06.2025

Synapse modeling refers to the process of complementing the simulated fibers with synaptic connections so that second-order effects of extracellular stimulations can be investigated. In this tutorial we are going to learn the following:

- How to tune the soma's excitatory post-synaptic potential

- How to add a motoneuron soma and initial segment to a motor fiber

- How to add synaptic events (inputs) on the soma

- How to use afferent action potentials as inputs to the motoneuron's soma and fiber

- How to run a full simulation of populations of fibers to quantify spinally-evoked reflexes and direct motor activations across different muscle groups

In [ ]:
import numpy as np
import h5py
# add parent directory to path
import sys
import os
from neuron import h
import time

# Add parent directory to path to import pyns modules
current_dir = os.path.dirname(os.path.abspath('__file__'))
pyns_root = os.path.join(current_dir, '..')
sys.path.insert(0, pyns_root)

from pyns.utils import filter_axon_trajectories, create_single_pulse_waveform
from pyns.axon_models import MyelinatedAxon, Motoneuron

import matplotlib.pyplot as plt

### Step 1: load the main resources for your simulation (field dict and axon fibers)
- This is the same step as in the first tutorial [PyNS_tutorial_0.ipynb](https://gitlab.rrze.fau.de/ProModell/pyns/-/blob/main/tutorials/PyNS_tutorial_0.ipynb).

In [ ]:
# Load the field dictionary from the test dataset
# Get pyns root directory
pyns_root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Path to test dataset files
field_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'lumbar-tSCS_cathode_T11-T12_anode_navel-sides_units_V_m_cropped.h5')
axons_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'RightSoleusAxons_diams_from_Schalow1992_cropped.npy')

# Load field dictionary from HDF5 file
with h5py.File(field_path, 'r') as f:
    field_dict = {key: f[key][()] for key in f.keys()}

# Convert coordinates from m to um for consistency with axon coordinates
field_dict["x"] *= 1e6  # m to um
field_dict["y"] *= 1e6  # m to um
field_dict["z"] *= 1e6  # m to um

print(f"Field dictionary entries:")
for key in field_dict.keys():
    print(f"  {key}: shape={field_dict[key].shape}, dtype={field_dict[key].dtype}")

# Define spatial ranges for filtering axon trajectories (with 1mm safety margin)
x_range = [field_dict["x"].min() + 1000, field_dict["x"].max() - 1000]
y_range = [field_dict["y"].min() + 1000, field_dict["y"].max() - 1000]
z_range = [field_dict["z"].min() + 1000, field_dict["z"].max() - 1000]

# Load and filter axon trajectories
fibers_dict = np.load(axons_path, allow_pickle=True)[()]
print(f"\nNumber of fibers before filtering: {len(fibers_dict)}")

# Minimum axon length: fibers shorter than this are unlikely to show meaningful responses
min_axon_length = 40.0 * 1e3  # in micrometers
axon_dicts = filter_axon_trajectories(
    fibers_dict,
    x_range=x_range,
    y_range=y_range,
    z_range=z_range,
    min_axon_length=min_axon_length,
    rank=0,
)
print(f"Number of fibers after filtering: {len(axon_dicts)}")

--------------------------------------------------------
### Step 2: Create a Motoneuron and tune its EPSP due to afferent inputs

For tuning we keep modifying the synaptic weight of our synapse until we get the desired EPSP. For a list of EPSPs reported in the literature please refer to the docstring of the function connect_to_source of the Motoneuron class.

In [ ]:
sim_dt = 0.005  # in ms
sim_dur = 5.0  # in ms

In [ ]:
dt = 0.005
tstop = 10.0
spike_t = 7.0
weight = 0.002556

# EPSP for a fiber diameter of 12.95 um
fiber_diam = 12.95
print(f"init_seg_diam: {fiber_diam*(3.5/6.4)}")
mn = Motoneuron(
    init_seg_diam=fiber_diam*(3.5/6.4)
)
mn.setup_recorders(record_v=True)
vecstims = []
axon_spike_times = [spike_t]
n_synapses = 1
for i in range(n_synapses):
    spike_times = np.copy(axon_spike_times)
    syn = mn.create_synapse(type="excitatory", x=0.5)
    spiketime_v = h.Vector(spike_times)				#Create a h.Vector() and insert the time element in the vector.
    vecstim = h.VecStim()
    vecstim.play(spiketime_v)				#Play the spiketime.
    vecstims.append(vecstim)
for syn_i, vecstim in enumerate(vecstims):
    nc = mn.connect_to_source(vecstim, synapses_index=syn_i, weight=weight)


h.celsius = 37
h.dt = dt
h.tstop = tstop
h.v_init = -70
# integrator
h.cvode_active(0)

init_hoc_path = "../src/pyns/init_diff_v.hoc"

# Initialize simulation
if init_hoc_path is not None:
    h.load_file(init_hoc_path)
else:
    h.finitialize()
    h.fcurrent()

# Run simulation
t1 = time.time()
h.run(h.tstop)
t2 = time.time()
print(f"Simulation time: {t2-t1:.2f} s")

mn.get_epsp(syn_t=spike_t, plot=True)

### Step 3: Connect the motoneuron to a motor fiber
Here we are going to create a motor axon from one of the ventral trajectories and connect it to a motoneuron.

In [ ]:
# Select a ventral motor axon
import time
axon_dict = [axd for axd in axon_dicts if "VR" in axd["axon_name"]][0]

# Create a motor fiber axon with motoneuron soma and initial segment
axon_obj = MyelinatedAxon(
    axon_name=axon_dict["axon_name"],
    axon_coords=np.copy(axon_dict["points"]),
    fiber_diameter=axon_dict["diam"],
    model_type="motor",  # Force motor model for efferent axons
    tuned_model=True,
    params_fit_method="continuous",
)

# Create a monophasic stimulus
stim_pulse_t, stim_pulse = create_single_pulse_waveform(
    start_at=1.0,
    end_at=3.0,
    amplitude=1.0,
    biphasic=False,
)

# Initialize the axon with motoneuron
axon_obj.initialize_neuron(
    passive_end_nodes=True,
    end_connected_to_mn=True,
)

# Setup recorders
axon_obj.setup_recorders(record_v=True)

# Assign extracellular voltage
axon_obj.assign_v_ext(field_dict)

# Create motoneuron connected to motor fiber
fiber_mn = Motoneuron(
    name=f"MN_{axon_obj.name}",
)
fiber_mn.setup_recorders(record_v=True)

# Run a test simulation
stim_factor = 8.0  # Scaling factor for the stimulus
axon_res = axon_obj.run_simulation(
    stim_factor=stim_factor,
    stim_pulse=stim_pulse,
    dt=sim_dt,
    tstop=sim_dur,
    output_path=None,
    return_only_spiking=False,
    exclude_end_node=False,
    prepassive_nodes_as_endnodes=False,
    delete_hoc_objects=False,
)

# Plot the axon results
print(f"Membrane potential of the motor axon:")
axon_obj.plot_membrane_potential()
plt.show()

# Clean up
axon_obj.delete_sections()
axon_obj.delete_recorders()

### Step 4: Connect afferent action potentials to the motoneuron
When running an actual `sPyNS` simulation. After simulating afferent fibers, the script calculates the initiation node of each of the action potentials along the afferents and also calculates the times of arrival for each of these action potentials at the motoneuron of interest. For details on this, please have a look at the functions `get_ap_init_nodes()` and `get_ap_times_at_mn()` in [simulation_utils.py](https://gitlab.rrze.fau.de/ProModell/pyns/-/blob/main/simulation_utils.py).

In the following example we will just create a fake dictionary of these spikes times as if they were calculated in the script.

In [ ]:
from pyns.sim_utils import discretize_and_interpolate_v_fiber
axon_dict = [axd for axd in axon_dicts if "VR" in axd["axon_name"]][0]

# now we use the descritization function to descritize the axon and interpolate extracellular voltage with the motoneuron flag
axon_obj_dict = discretize_and_interpolate_v_fiber(
        axon_info=axon_dict,
        model_type=None,
        tuned_flag=False,
        field_dict=field_dict,
        motoneuron=True
    )

# create a monophasic stimulus
stim_t, stim_pulse = create_single_pulse_waveform(
    start_at=1.0,
    end_at=3.0,
    time_step=sim_dt,
)

# for each fiber, generate list of 2 random spike times
n_afferent_fibers = 40  # Number of afferent fibers
afferent_input_dict = {}
for i in range(1, n_afferent_fibers + 1):
    fiber_name = f"aff_fiber{i}"
    # Generate two random spike times within the simulation duration
    spike_times = np.random.uniform(0, 2, size=2).tolist()
    afferent_input_dict[fiber_name] = spike_times

# Now create a new motoneuron and set the afferent inputs
# Create the fiber object
axon_obj = MyelinatedAxon(
    discretized_dict=axon_obj_dict,
)
# Initialize the axon object (creates sections, nodes, and recorders)
axon_obj.initialize_neuron(
    passive_end_nodes=True,
    end_connected_to_mn=True
)
# Setup recorders for the axon object
axon_obj.setup_recorders(record_v=True)
# Assign the extracellular voltage to the axon sections
axon_obj.assign_v_ext()

# Now create the motoneuron and connect it to the axon object
fiber_mn = Motoneuron(
    name=f"MN_{axon_obj.name}",
    init_seg_diam=axon_obj.axonD*(3.5/6.4), # dima ratio from Cullheim and Kellerth, 1978
    soma_coord=axon_obj_dict["mn_soma_coord"],
    initseg_coord=axon_obj_dict["mn_initseg_coord"],
    )
fiber_mn.setup_recorders(record_v=True)
fiber_mn.initSegment.connect(axon_obj.sections_list[-1], 1, 1)
fiber_mn.assign_v_ext(
    axon_obj_dict["mn_soma_v"],
    axon_obj_dict["mn_initseg_v"],
)
fiber_mn.set_Ia_afferent_inputs(
    afferent_input_dict,
    syn_weight=0.008,
)

# Run a test simulation
stim_factor = 50.0  # This is the factor by which the stimulus is scaled.
axon_res = axon_obj.run_simulation(
    stim_factor=stim_factor,
    stim_pulse=stim_pulse,
    dt=sim_dt,
    tstop=sim_dur,
    output_path=None,
    return_only_spiking=False,
    exclude_end_node=False,
    prepassive_nodes_as_endnodes=False,
    delete_hoc_objects=False,
    # init_hoc_path=init_hoc_path,
)

# Plot the motoneuron results
print(f"Membrane potential of the soma and initial segment of the motoneuron:")
fiber_mn.plot_membrane_potential()
print(f"Membrane potential of the axon nodes:")
axon_obj.plot_membrane_potential()
axon_obj.delete_sections()
axon_obj.delete_recorders()

The response above can be called afferent-initiated response, since it was induced by the synapse. If, however, it was a direct activation of the efferent fiber, we call it efferent-initiated response. Try increasing the stimulation factor and see at which point you start getting only efferent-initiated response.

When running an actual `sPyNS` simulation. The script classifies responses observed on the efferents and stores the result so that later they can be processed easily knowing which was afferent-initiated (reflex) response and which is efferent initated (direct) response. Please refer to the README to run the an example on the provided test dataset.

In [ ]:
!pyns-run-discrete-simulations --help